# RAPTOR

Recursively cluster and summarise chunks into a tree; retrieve across all levels.

**Extra requirements:** `pip install scikit-learn`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def build_tree(leaf_chunks, max_levels=3, cluster_size=4):
    levels = [leaf_chunks]
    for _ in range(max_levels):
        current = levels[-1]
        if len(current) <= cluster_size:
            break
        n_clusters = max(2, len(current) // cluster_size)
        labels = KMeans(n_clusters=n_clusters, n_init="auto", random_state=0).fit_predict(embed(current))
        summaries = []
        for c in range(n_clusters):
            members = [t for t, l in zip(current, labels) if l == c]
            summaries.append(llm("Summarise these passages in 3 sentences:\n\n" + "\n\n".join(members)))
        levels.append(summaries)
    return levels

tree = build_tree(chunks)
print([len(level) for level in tree], "nodes per level (leaves first)")

## Collapsed-tree retrieval
Put every node from every level into one index. Detail questions hit leaves; broad questions hit summaries.

In [ ]:
nodes, meta = [], []
for depth, level in enumerate(tree):
    nodes += level; meta += [{"level": depth}] * len(level)
tree_store = build_store(nodes, meta)

for q in ["What is the dimension of all-MiniLM-L6-v2 vectors?", "What are the major topics covered in this handbook?"]:
    hits = tree_store.search(q, 3)
    print(q, "-> levels hit:", [h["meta"]["level"] for h in hits])
    print(answer(q, [h["text"] for h in hits])[:300], "\n")